# 03 - Water-type OWT from invRrs, from oligotrophic to extremely turbid (replacement of Cordeiro 2022)

GRSl2bgen uses the 4 SPM classes of Cordeiro (2022) (`owt_cordeiro2022_spm.nc`: clear, moderate, turbid, very
turbid; Euclidean distance on Rrs) to blend the SPM algorithms. They are replaced here by a series of water types
simulated with the invRrs forward model (`TwoModesSi`, relationship `poly2_fd` of the article *Hyperspectral
relationships between IOPs and reflectance ... clear to extremely turbid waters*), extending the water types of
Fig. 9 of `invRrs/notebooks/article/forward_model_article.ipynb`: oligotrophic, mesotrophic, productive,
blooms, CDOM-rich, turbid and extremely turbid waters.

Each type is an ensemble (log-normal spread of the concentrations around the type), so that a class has a
median spectrum and a spread like an OWT derived from data. The library keeps the magnitude (sr$^{-1}$): it can
be used with the Euclidean distance (as Cordeiro, magnitude matters for SPM) or with the spectral angle.

Outputs: `data/owt_water_types_invRrs_hyper.nc` and `data/owt_water_types_invRrs_s2.nc`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

from ta_common import *

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'font.size': 10})

In [ ]:
import time
from invRrs import solver, utils

WL = np.arange(400., 1001., 5.)
RHO = 2.65
geom = json.loads((DATA / f'geometry_{TILE}.json').read_text()) if (DATA / f'geometry_{TILE}.json').exists() else {}
SZA, VZA, AZI = round(geom.get('sza_median_mar_sep', 35.0)), 5., 100.
aphy_star, bbphy_star = utils.data().load_iop_phyto('Ciotti2002_Sf03')
model = solver.two_modes_Si(aphy_star, bbphy_star, sza=SZA, wl_=WL, level=1, iop_var='bssa', model='poly2_fd')
model.set_coef(sza=SZA, vza=VZA, azi=AZI)
print(f'sza {SZA}, vza {VZA}, azimuth {AZI}; parameters: chl, a_bg(440), S_bg, SPM/rho, p_f, n_r, n_i,ref, S_ni')

## 1. Water types

Parameters of `TwoModesSi.forward_model`: Chl-a (mg m$^{-3}$), $a_{bg}(440)$ (CDOM + detritus, m$^{-1}$), $S_{bg}$
(nm$^{-1}$), SPM (g m$^{-3}$, divided by the density 2.65), fine-mode proportion $p_f$, real and imaginary refractive
index of the sediment at 440 nm, spectral slope of $n_i$. The first six types are those of the invRrs article
(Fig. 9); the others fill the gaps along the three axes (biomass, CDOM, sediment). `SPREAD` is the log-normal
spread (1 sigma, factor) of Chl, $a_{bg}$ and SPM in the ensemble of each type. `SPM_ALGO` is the SPM algorithm
proposed for each class for the OWT blending of GRSl2bgen (to be confirmed: it replaces the Cordeiro recipe).

In [ ]:
WATER_TYPES = {  # name: chl, a_bg440, S_bg, SPM, p_f, n_r, n_i_ref, S_ni
    'WT01 oligotrophic':            [0.1, 0.005, 0.015, 0.05, 0.8, 1.17, -0.001, 0.01],
    'WT02 mesotrophic open sea':    [0.5, 0.03, 0.015, 0.3, 0.7, 1.17, -0.002, 0.01],
    'WT03 mesotrophic coastal':     [2, 0.1, 0.015, 2, 0.6, 1.17, -0.003, 0.01],
    'WT04 eutrophic coastal':       [8, 0.3, 0.015, 4, 0.5, 1.17, -0.003, 0.01],
    'WT05 productive / bloom':      [30, 0.3, 0.015, 5, 0.5, 1.17, -0.003, 0.01],
    'WT06 phytoplankton bloom':     [80, 0.3, 0.015, 5, 0.5, 1.17, -0.003, 0.01],
    'WT07 hypereutrophic':          [300, 0.6, 0.015, 8, 0.5, 1.17, -0.003, 0.01],
    'WT08 CDOM-rich coastal':       [2, 0.8, 0.017, 3, 0.5, 1.17, -0.003, 0.01],
    'WT09 CDOM-rich lake':          [5, 1.5, 0.017, 3, 0.5, 1.17, -0.003, 0.01],
    'WT10 dark CDOM (black water)': [3, 6.0, 0.018, 2, 0.5, 1.17, -0.003, 0.01],
    'WT11 moderately turbid':       [3, 0.2, 0.015, 15, 0.5, 1.17, -0.004, 0.01],
    'WT12 turbid estuary':          [5, 0.3, 0.015, 100, 0.5, 1.17, -0.005, 0.01],
    'WT13 very turbid':             [5, 0.4, 0.015, 300, 0.45, 1.17, -0.006, 0.01],
    'WT14 extremely turbid river':  [5, 0.5, 0.015, 800, 0.4, 1.17, -0.008, 0.01],
}
SPM_ALGO = {'WT01': 'Nechad 665', 'WT02': 'Nechad 665', 'WT03': 'Nechad 665', 'WT04': 'Nechad 665',
            'WT05': 'Nechad 665', 'WT06': 'Nechad 665', 'WT07': 'Nechad 705', 'WT08': 'Nechad 665',
            'WT09': 'Nechad 665', 'WT10': 'Nechad 665', 'WT11': 'Nechad 705', 'WT12': 'Nechad 783',
            'WT13': 'Nechad 865', 'WT14': 'Novoa / Dogliotti 865'}      # TO BE CONFIRMED
CORDEIRO_OF = {'WT01': 1, 'WT02': 1, 'WT03': 1, 'WT04': 2, 'WT05': 2, 'WT06': 2, 'WT07': 2, 'WT08': 1,
               'WT09': 2, 'WT10': 1, 'WT11': 3, 'WT12': 3, 'WT13': 4, 'WT14': 4}  # expected Cordeiro class
SPREAD = 1.5             # factor (1 sigma) on chl, a_bg, SPM
N_ENS = 200

In [ ]:
rng = np.random.default_rng(0)
t0 = time.time()
ens = np.full((len(WATER_TYPES), N_ENS, len(WL)), np.nan)
center = np.full((len(WATER_TYPES), len(WL)), np.nan)
for i, (name, p) in enumerate(WATER_TYPES.items()):
    p = np.array(p, float)
    q = p.copy(); q[3] /= RHO
    center[i] = model.forward_model(list(q), output='Rrs').values
    for j in range(N_ENS):
        q = p.copy()
        q[[0, 1, 3]] *= SPREAD ** rng.standard_normal(3)
        q[3] /= RHO
        ens[i, j] = model.forward_model(list(q), output='Rrs').values
print(f'{ens.shape[0] * ens.shape[1]} spectra in {time.time() - t0:.0f} s, non-finite: {np.isnan(ens).sum()}')
names = list(WATER_TYPES)
codes = [n.split()[0] for n in names]
hyper = xr.Dataset({'Rrs_center': (('owt', 'wl'), center), 'Rrs_ens': (('owt', 'member', 'wl'), ens)},
                   coords={'owt': codes, 'wl': WL, 'member': range(N_ENS), 'label': ('owt', names)})
hyper['m_Rrs'] = hyper.Rrs_ens.median('member')
hyper['lo_Rrs'] = hyper.Rrs_ens.quantile(0.1, 'member').drop_vars('quantile')
hyper['up_Rrs'] = hyper.Rrs_ens.quantile(0.9, 'member').drop_vars('quantile')

In [ ]:
cmap = plt.cm.Spectral_r
WT_COLOR = {c: mpl.colors.to_hex(cmap(x)) for c, x in zip(codes, np.linspace(0, 1, len(codes)))}
fig, axs = plt.subplots(1, 2, figsize=(15, 5.5))
for c, n in zip(codes, names):
    for ax in axs:
        ax.fill_between(WL, hyper.lo_Rrs.sel(owt=c), hyper.up_Rrs.sel(owt=c), color=WT_COLOR[c], alpha=0.15, lw=0)
        ax.plot(WL, hyper.m_Rrs.sel(owt=c), color=WT_COLOR[c], lw=1.8, label=n)
axs[1].set_yscale('log'); axs[1].set_ylim(1e-5, 0.2)
for ax in axs:
    ax.set(xlabel='wavelength (nm)', ylabel='Rrs (sr$^{-1}$)')
    for b in BANDS:
        ax.axvline(b, color='0.9', lw=0.6, zorder=0)
axs[0].legend(fontsize=7, frameon=False)
axs[0].set_title(f'invRrs water types: median and 10-90 % of {N_ENS} members (spread x{SPREAD})')
fig.tight_layout()
fig.savefig(FIG / 'water_types_invRrs_hyper.png')

## 2. Library on the MSI bands

In [ ]:
s2_med = s2_convolution(WL, hyper.m_Rrs.values)
s2_ens = s2_convolution(WL, hyper.Rrs_ens.values)
lib = xr.DataArray(s2_med, coords={'owt': codes, 'wl': BANDS}, dims=('owt', 'wl'), name='m_Rrs')
lib = lib.assign_coords(label=('owt', names), color=('owt', [WT_COLOR[c] for c in codes]),
                        group=('owt', ['biomass'] * 7 + ['CDOM'] * 3 + ['sediment'] * 4),
                        spm_algo=('owt', [SPM_ALGO[c] for c in codes]))
lib.attrs.update(units='sr-1', long_name='median above-water Rrs of the invRrs water types')
write_owt(lib, WT_OWT_FILE, title='invRrs optical water types, oligotrophic to extremely turbid, MSI bands',
          model='invRrs TwoModesSi poly2_fd, level 1 (above water)', sza=SZA, vza=VZA, azi=AZI,
          parameters=json.dumps(WATER_TYPES), spread=SPREAD, n_members=N_ENS, srf='S2A (Py6S)')
hyper.attrs.update(title='invRrs optical water types, hyperspectral', parameters=json.dumps(WATER_TYPES))
hyper.to_netcdf(WT_OWT_HYPER_FILE)
s2_ens_da = xr.DataArray(s2_ens, coords={'owt': codes, 'member': range(N_ENS), 'wl': BANDS}, dims=('owt', 'member', 'wl'))
s2_ens_da.to_netcdf(DATA / 'water_types_invRrs_s2_ensemble.nc')
lib.to_pandas().map(lambda v: f'{v:.2e}')

## 3. Self-consistency and comparison with Cordeiro (2022) and Bi (2024)

The ensemble members are classified against the library medians with both distances: the share of members
returned to their own type measures the separability of the types (the spread being arbitrary, these numbers
are relative). The members are also classified with Cordeiro (Euclidean, as in GRSl2bgen) and with the Bi et al.
(2024) types (spectral angle) to see how the new series maps on the current classes.

In [ ]:
def euclid_class(R, ref):
    d = np.sqrt(((R[:, None, :] - ref[None]) ** 2).sum(-1))
    return d.argmin(1) + 1


flat = s2_ens.reshape(-1, len(BANDS))
truth = np.repeat(np.arange(1, len(codes) + 1), N_ENS)
b7 = [BANDS.index(b) for b in BANDS_GRSL2BGEN]
pred_euc = euclid_class(flat[:, b7], lib.values[:, b7])
pred_sam = sam_topk(flat[:, b7], lib.values[:, b7], k=1)[0][:, 0]
cord = load_owt(CORDEIRO_FILE)
cord_s2 = cord.interp(wl=BANDS)
pred_cord = euclid_class(flat[:, b7], cord_s2.values[:, b7])
bi = load_owt(TA25_V1_FILE).isel(owt=slice(0, 10)).interp(wl=BANDS)
pred_bi = sam_topk(flat[:, b7], bi.values[:, b7], k=1)[0][:, 0]
print(f'members back to their own type: Euclidean {np.mean(pred_euc == truth):.0%}, spectral angle {np.mean(pred_sam == truth):.0%}')

fig, axs = plt.subplots(1, 4, figsize=(21, 5.5))
for ax, (pred, labs, t) in zip(axs, [(pred_euc, codes, 'invRrs types, Euclidean'), (pred_sam, codes, 'invRrs types, SAM'),
                                     (pred_cord, list(cord.owt.values), 'Cordeiro 2022, Euclidean'),
                                     (pred_bi, list(bi.owt.values), 'Bi 2024, SAM')]):
    cm = pd.crosstab(pd.Categorical(np.array(codes)[truth - 1], codes),
                     pd.Categorical(np.array(labs)[pred.astype(int) - 1], labs), normalize='index', dropna=False)
    ax.imshow(cm.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(labs)), labs, rotation=90, fontsize=7); ax.set_yticks(range(len(codes)), names, fontsize=7)
    ax.set_title(t, fontsize=10)
fig.tight_layout()
fig.savefig(FIG / 'water_types_invRrs_confusion.png')
pd.DataFrame({'expected Cordeiro': [CORDEIRO_OF[c] for c in codes],
              'Cordeiro (members, mode)': [pd.Series(pred_cord[truth == i + 1]).mode().iloc[0] for i in range(len(codes))],
              'Bi 2024 (members, mode)': [bi.owt.values[int(pd.Series(pred_bi[truth == i + 1]).mode().iloc[0]) - 1] for i in range(len(codes))],
              'SPM algorithm': [SPM_ALGO[c] for c in codes]}, index=names)

In [ ]:
# the water types and the HABLAB blooms (02) together: are the bloom types separable from WT05-WT07?
if HABLAB_OWT_FILE.exists():
    hab = load_owt(HABLAB_OWT_FILE)
    a = np.degrees(angles(hab.sel(wl=BANDS_GRSL2BGEN).values, lib.sel(wl=BANDS_GRSL2BGEN).values))
    near = pd.DataFrame({'closest water type': [codes[i] for i in a.argmin(1)], 'angle (deg)': a.min(1).round(1)},
                        index=hab.owt.values)
    display(near.T)